# Clase 198 — Docker para empaquetar modelos

Notebook declarativo: genera el `Dockerfile`, `.dockerignore`, `app.py`, `requirements.txt` en un directorio temporal. Las celdas de `docker build/run` se muestran como shell commands — requieren Docker instalado para ejecutarse.

## Setup — generar el proyecto

In [ ]:
import os, shutil, tempfile
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'docker_demo'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(); os.chdir(WORK)

# 1. Entrenar un modelo y guardarlo
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
X, y = load_iris(return_X_y=True)
m = RandomForestClassifier(n_estimators=50, random_state=42).fit(X, y)
joblib.dump(m, 'model.pkl')
print('model.pkl:', Path('model.pkl').stat().st_size, 'bytes')

In [ ]:
Path('app.py').write_text('''\
from fastapi import FastAPI
from pydantic import BaseModel
import joblib, numpy as np

model = joblib.load("model.pkl")
app = FastAPI()

class In(BaseModel):
    features: list[float]

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/predict")
def predict(x: In):
    pred = int(model.predict(np.array(x.features).reshape(1, -1))[0])
    return {"class": pred}
''')

Path('requirements.txt').write_text('fastapi==0.115.0\nuvicorn[standard]==0.32.0\nscikit-learn==1.5.2\njoblib==1.4.2\n')
print('app.py + requirements.txt listos')

## 1. Dockerfile naive (mal hecho a propósito)

In [ ]:
bad = '''\
FROM python:3.12
COPY . /app
WORKDIR /app
RUN pip install -r requirements.txt
CMD uvicorn app:app --host 0.0.0.0 --port 8000
'''
Path('Dockerfile.bad').write_text(bad)
print(bad)
print('# Problemas: imagen base full (~1 GB), COPY antes de RUN pip (cache-busting),')
print('# corre como root, sin healthcheck, sin .dockerignore.')

## 2. Dockerfile multi-stage correcto

In [ ]:
good = '''\
# --- Stage 1: build wheels (incluye compiladores) ---
FROM python:3.12-slim AS builder
WORKDIR /build
RUN pip install --no-cache-dir --upgrade pip wheel
COPY requirements.txt .
RUN pip wheel --no-cache-dir --wheel-dir /wheels -r requirements.txt

# --- Stage 2: runtime slim ---
FROM python:3.12-slim AS runtime

RUN groupadd -r app && useradd -r -g app -u 1000 -m app
WORKDIR /app

COPY --from=builder /wheels /wheels
COPY requirements.txt .
RUN pip install --no-cache-dir --no-index --find-links=/wheels -r requirements.txt \\
    && rm -rf /wheels

COPY --chown=app:app app.py model.pkl ./
USER app

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')" || exit 1
ENTRYPOINT ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''
Path('Dockerfile').write_text(good)
print(good)

## 3. `.dockerignore`

In [ ]:
Path('.dockerignore').write_text('''\
.git
.gitignore
__pycache__
*.pyc
*.ipynb
.ipynb_checkpoints
data/
mlruns/
*.md
tests/
''')
print(Path('.dockerignore').read_text())

## 4. Build, run, inspect (requiere Docker)

In [ ]:
# Comentado por default — descomentá si tenés Docker corriendo localmente.
# !docker build -t iris-api:v1 .
# !docker images iris-api:v1 --format 'TAG={{.Tag}} SIZE={{.Size}}'
# !docker run -d --name iris -p 8000:8000 iris-api:v1
# import time; time.sleep(2)
# !curl -s -X POST localhost:8000/predict -H 'content-type: application/json' -d '{"features":[5.1,3.5,1.4,0.2]}'
# !docker exec iris whoami   # debe decir "app", no "root"
# !docker history iris-api:v1 --human --format 'table {{.CreatedBy}}\t{{.Size}}'
# !docker stop iris && docker rm iris

print('Para correr esta sección, instalá Docker Desktop / Docker Engine y descomentá las líneas.')

## Ejercicio guiado

1. Buildeá `Dockerfile.bad` y `Dockerfile`. Compará tamaños (`docker images`). El multi-stage debería ser <300 MB; el naive >1 GB.
2. Cambiá una línea en `app.py` y rebuildeá ambos. Confirmá que el bueno solo reconstruye la última capa, el malo reconstruye todo.
3. Corré `trivy image iris-api:v1 --severity HIGH,CRITICAL`. Anotá las CVEs y proponé fixes (bump de base, `apt-get update`).
4. Pushá la imagen a Docker Hub (`docker push <user>/iris-api:1.0.0`) y obtené el digest. Reemplazá `:1.0.0` por `@sha256:...` en tu deploy.

## Conclusiones

- `python:3.12-slim` + multi-stage corta imagen 4× sin perder funcionalidad.
- Orden de instrucciones = velocidad de rebuild.
- Non-root + healthcheck son requisito mínimo, no "nice to have".
- Producción referencia **digest**, no `:latest`.